In [2]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga de los ficheros con la distribución por fuente de ingresos

Se han estructurado los datos en la carpeta de inputs para la dimensión socioeconómica de forma que se dispone de un fichero CSV a nivel
de provincia. Así pues, se han definido una función en las utilidades que se encargan de cargar y dar una limpieza inicial a los datos.

Los datos se obtienen del atlas de distribución de renta de los hogares:
https://www.ine.es/dynt3/inebase/index.htm?padre=12385&capsel=12384

In [3]:
path = os.path.join(DATA_INPUTS_DS, "Distribución por fuente de ingresos")

indicadores = carga_datos_ine(path)

# Veo una muestra de su estructura y contenido
print(indicadores.info())
indicadores.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 160695 entries, 90 to 368279
Data columns (total 7 columns):
 #   Column                               Non-Null Count   Dtype 
---  ------                               --------------   ----- 
 0   Municipios                           160695 non-null  object
 1   Distritos                            160695 non-null  object
 2   Secciones                            160695 non-null  object
 3   Distribución por fuente de ingresos  160695 non-null  object
 4   Periodo                              160695 non-null  int64 
 5   Total                                127370 non-null  object
 6   Provincia                            160695 non-null  object
dtypes: int64(1), object(6)
memory usage: 9.8+ MB
None


,Municipios,Distritos,Secciones,Distribución por fuente de ingresos,Periodo,Total,Provincia
188816,37152 Gomecello,3715201 Gomecello distrito 01,3715201001 Gomecello sección 01001,Fuente de ingreso: otros ingresos,2018,"9,4",Salamanca
278447,42173 Soria,4217303 Soria distrito 03,4217303001 Soria sección 03001,Fuente de ingreso: otras prestaciones,2018,"3,2",Soria
143604,34048 Castil de Vela,3404801 Castil de Vela distrito 01,3404801001 Castil de Vela sección 01001,Fuente de ingreso: pensiones,2023,NaN,Palencia
53378,09071 Carcedo de Bureba,0907101 Carcedo de Bureba distrito 01,0907101001 Carcedo de Bureba sección 01001,Fuente de ingreso: salario,2015,NaN,Burgos
305391,47139 Rueda,4713901 Rueda distrito 01,4713901001 Rueda sección 01001,Fuente de ingreso: prestaciones por desempleo,2020,"4,9",Valladolid


# Estandarización del dataframe de datos del INE

Como se puede observar, el fichero csv de datos del INE tiene un formato poco amigable para el tratamiento de los datos. En lugar de tener una fila
por cada par sección-año y varias columnas (una por factor), tiene múltiples filas con distintos indicadores para una misma sección, lo que resulta
complejo de tratar. Además, se observa como se mezcla el código del municipio, distrito y seccion con el texto, y deberían tener una columna con
los códigos.

In [7]:
indicadores_estandarizados = estandarizar_df_ine(indicadores, "Distribución por fuente de ingresos")
print(indicadores_estandarizados.info())
indicadores_estandarizados.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 160695 entries, 90 to 368279
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   Provincia  160695 non-null  object
 1   CMuni      160695 non-null  object
 2   CUSEC      160695 non-null  object
 3   Indicador  160695 non-null  object
 4   Periodo    160695 non-null  int64 
 5   Total      127370 non-null  object
dtypes: int64(1), object(5)
memory usage: 8.6+ MB
None


,Provincia,CMuni,CUSEC,Indicador,Periodo,Total
230421,Segovia,40039,4003901001,Fuente de ingreso: prestaciones por desempleo,2020,"2,7"
156169,Palencia,34130,3413001001,Fuente de ingreso: prestaciones por desempleo,2022,"1,4"
241375,Segovia,40125,4012501001,Fuente de ingreso: otros ingresos,2019,NaN
318968,Valladolid,47186,4718608017,Fuente de ingreso: salario,2015,"46,6"
289977,Valladolid,47040,4704001001,Fuente de ingreso: otros ingresos,2017,NaN


## Filtrado de años
Revisando la documentación del INE, en el año 2021 se cambió radicalemente la metodología que define las secciones censales,
y en concreto en Castilla y León se aumentó el numero de censos de 2700 a unos 3500 apróximadamente. Es por ello que, si bien
se dispone de datos de años anteriores, sería complejo y peligroso fragmentar y proyectar los censos de años previos en la malla 
censal actual, por lo que se filtraran datos de años previos

In [8]:
# Reviso los indicadores disponibles
revisar_indicadores_disponibles(indicadores_estandarizados)

# Filtro por los años 2021 - 2023.
indicadores_recientes = indicadores_estandarizados[
    indicadores_estandarizados["Periodo"].isin([2021, 2022, 2023])
].copy()

📅 Años disponibles:
[2023 2022 2021 2020 2019 2018 2017 2016 2015]
------------------------------------------------------------
🧩 Indicadores demográficos disponibles:
  - Fuente de ingreso: salario
  - Fuente de ingreso: pensiones
  - Fuente de ingreso: prestaciones por desempleo
  - Fuente de ingreso: otras prestaciones
  - Fuente de ingreso: otros ingresos
------------------------------------------------------------


In [9]:
# Pivoto los indicadores para tener una columna por indicador y reducir las filas de la tabla
indicadores_por_seccion = pivotar_indicadores(indicadores_recientes)
print(indicadores_por_seccion.info())
indicadores_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8390 entries, 0 to 8389
Data columns (total 9 columns):
 #   Column                                        Non-Null Count  Dtype 
---  ------                                        --------------  ----- 
 0   Provincia                                     8390 non-null   object
 1   CMuni                                         8390 non-null   object
 2   CUSEC                                         8390 non-null   object
 3   Periodo                                       8390 non-null   int64 
 4   Fuente_de_ingreso_otras_prestaciones          8390 non-null   object
 5   Fuente_de_ingreso_otros_ingresos              8390 non-null   object
 6   Fuente_de_ingreso_pensiones                   8390 non-null   object
 7   Fuente_de_ingreso_prestaciones_por_desempleo  8390 non-null   object
 8   Fuente_de_ingreso_salario                     8390 non-null   object
dtypes: int64(1), object(8)
memory usage: 590.1+ KB
None


,Provincia,CMuni,CUSEC,Periodo,Fuente_de_ingreso_otras_prestaciones,Fuente_de_ingreso_otros_ingresos,Fuente_de_ingreso_pensiones,Fuente_de_ingreso_prestaciones_por_desempleo,Fuente_de_ingreso_salario
5428,Segovia,40149,4014901001,2022,"2,6","31,2","29,8","0,7","35,7"
2612,Leon,24115,2411501009,2021,"5,7","3,9","23,7","3,8","62,9"
1289,Burgos,09114,0911401001,2022,"5,7","6,4","27,6","2,2","58,1"
5988,Soria,42173,4217303002,2023,"4,3",7,"20,2","1,7","66,8"
4679,Salamanca,37274,3727404019,2022,"5,8","4,2","23,5","2,3","64,2"


# Export de los resultados

In [10]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DS, exist_ok=True)

# Rutas de salida
ruta_seccion = os.path.join(DATA_OUTPUTS_DS, "fuente_ingresos_por_seccion.csv")

# Guardar DataFrames
indicadores_por_seccion.to_csv(
    ruta_seccion,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

print(f"✅ Archivos guardados correctamente en: {DATA_OUTPUTS_DS}")

✅ Archivos guardados correctamente en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DS_Dim_socioeconomica
